## Download and Cleaning

In [ ]:
import os
from pathlib import Path
import math
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ── ChEMBL Client ─────────────────────────────────────────────────────────────
from chembl_webresource_client.new_client import new_client

# ── RDKit ─────────────────────────────────────────────────────────────────────
from rdkit import Chem, RDLogger
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog('rdApp.error')

# ==============================================================================
# 1) CONFIGURATION
# ==============================================================================
BASE_DIR = Path(__file__).resolve().parent
TARGET_BASE_DIR = BASE_DIR

ASSAY_TYPE_TO_PROCESS = "IC50"
USE_CACHE_ONLY = False # Set to True if you only want to process existing raw files
UNIT_TO_M = {"PM": 1e-12, "NM": 1e-9, "UM": 1e-6, "MM": 1e-3, "M": 1.0, "µM": 1e-6, "μM": 1e-6}

CHEMBL_TARGET_IDS = [
    "CHEMBL3105", "CHEMBL1824", "CHEMBL4005", "CHEMBL3130", "CHEMBL3267",
    "CHEMBL3145", "CHEMBL4282", "CHEMBL2842", "CHEMBL3650", "CHEMBL2742",
    "CHEMBL1871", "CHEMBL203", "CHEMBL1957", "CHEMBL4630", "CHEMBL279",
    "CHEMBL267", "CHEMBL4722", "CHEMBL2185", "CHEMBL325", "CHEMBL1865",
]

# ==============================================================================
# 2) CHEMICAL STANDARDIZATION (RDKit Compat)
# ==============================================================================
remover = SaltRemover()
_CLEANUP_PARAMS = getattr(rdMolStandardize, "CleanupParameters", lambda: None)()

# Handling different RDKit versions for Normalizer/Reionizer/FragmentChooser
if hasattr(rdMolStandardize, "Normalizer"):
    _normalizer_obj = rdMolStandardize.Normalizer()
    def _normalize_mol(m): return _normalizer_obj.normalize(m)
elif hasattr(rdMolStandardize, "Normalize"):
    def _normalize_mol(m):
        try:
            return rdMolStandardize.Normalize(m, _CLEANUP_PARAMS)
        except TypeError:
            return rdMolStandardize.Normalize(m)
else:
    def _normalize_mol(m): return m

if hasattr(rdMolStandardize, "Reionizer"):
    _reionizer_obj = rdMolStandardize.Reionizer()
    def _reionize_mol(m): return _reionizer_obj.reionize(m)
elif hasattr(rdMolStandardize, "Reionize"):
    def _reionize_mol(m): return rdMolStandardize.Reionize(m)
else:
    def _reionize_mol(m): return m

if hasattr(rdMolStandardize, "LargestFragmentChooser"):
    _largest_frag = rdMolStandardize.LargestFragmentChooser()
    def _choose_largest_frag(m): return _largest_frag.choose(m)
else:
    def _choose_largest_frag(m):
        frags = Chem.GetMolFrags(m, asMols=True, sanitizeFrags=False)
        return max(frags, key=lambda x: x.GetNumAtoms()) if frags else m

if hasattr(rdMolStandardize, "TautomerEnumerator"):
    _taut = rdMolStandardize.TautomerEnumerator()
    def _canonicalize_taut(m):
        return _taut.Canonicalize(m) if hasattr(_taut, "Canonicalize") else _taut.canonicalize(m)
else:
    def _canonicalize_taut(m): return m

def standardize_molecule(smiles: str) -> Chem.Mol | None:
    if pd.isna(smiles):
        return None
    try:
        m = Chem.MolFromSmiles(smiles)
        if m is None:
            return None
        m = _choose_largest_frag(m)
        m = _normalize_mol(m)
        m = _reionize_mol(m)
        m = _canonicalize_taut(m)
        Chem.SanitizeMol(m)
        return m
    except Exception:
        return None

def mol_to_clean_smiles(m: Chem.Mol) -> str | None:
    try:
        return Chem.MolToSmiles(m, canonical=True)
    except Exception:
        return None

# ==============================================================================
# 3) QUALITY / pActivity / AGGREGATION
# ==============================================================================
def to_pActivity(row) -> float | None:
    pv = row.get("pchembl_value", np.nan)
    if pd.notna(pv):
        try:
            v = float(pv)
            if math.isfinite(v):
                return v
        except Exception:
            pass
    sv = row.get("standard_value", np.nan)
    su = row.get("standard_units", None)
    if pd.notna(sv) and su is not None:
        try:
            su = str(su).upper().strip()
            factor = UNIT_TO_M.get(su, None)
            if factor is None:
                return None
            v_m = float(sv) * factor
            if v_m <= 0:
                return None
            return -math.log10(v_m)
        except Exception:
            return None
    return None

def quality_filter(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["standard_type", "assay_type", "standard_units", "standard_relation"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.upper().str.strip()
    keep = pd.Series(True, index=df.index)
    keep &= (df.get("standard_type", "IC50") == ASSAY_TYPE_TO_PROCESS)
    if "standard_flag" in df.columns:
        keep &= (df["standard_flag"].fillna(1) == 1)
    if "potential_duplicate" in df.columns:
        keep &= (df["potential_duplicate"].fillna(0) == 0)
    if "data_validity_comment" in df.columns:
        keep &= df["data_validity_comment"].isna()
    if "assay_confidence_score" in df.columns:
        keep &= (df["assay_confidence_score"].fillna(0) >= 8)
    if "assay_type" in df.columns:
        keep &= df["assay_type"].isin(["B"])
    keep &= df["canonical_smiles"].notna()
    keep &= df["standard_value"].notna() | df["pchembl_value"].notna()
    return df[keep]

def annotate_censor(row):
    rel = row.get("standard_relation", "=")
    if pd.isna(rel): rel = "="
    rel = str(rel).strip()
    if rel in (">", ">="): return "right_censored"
    if rel in ("<", "<="): return "left_censored"
    return "exact"

def robust_mad(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return float(np.median(np.abs(x - med)))

def preprocess_to_pIC50_tables(reports: pd.DataFrame):
    qc = {}
    df = quality_filter(reports)
    qc["n_raw"] = len(reports)
    qc["n_after_quality"] = len(df)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), qc

    df["censor"] = df.apply(annotate_censor, axis=1)
    qc["censor_counts"] = df["censor"].value_counts(dropna=False).to_dict()

    df["pActivity"] = df.apply(to_pActivity, axis=1)
    df = df[df["pActivity"].notna()]
    qc["n_with_pActivity"] = len(df)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), qc

    # Standardize Molecules
    df["Mol"] = [standardize_molecule(smi) for smi in df["canonical_smiles"].tolist()]
    df = df[df["Mol"].notna()]
    qc["n_after_standardize"] = len(df)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), qc

    df["std_canonical_smiles"] = df["Mol"].map(mol_to_clean_smiles)

    # Filter for exact matches
    df_exact = df[df["censor"] == "exact"].copy()
    qc["n_exact"] = len(df_exact)
    if df_exact.empty:
        return pd.DataFrame(), pd.DataFrame(), qc

    if "assay_id" not in df_exact.columns:
        df_exact["assay_id"] = "NA"

    # Aggregation per Assay
    per_assay = (df_exact
        .groupby(["molecule_chembl_id", "assay_id", "std_canonical_smiles"], as_index=False)
        .agg(pIC50=("pActivity", "median"),
             n_repl=("pActivity", "size"),
             iqr=("pActivity", lambda x: x.quantile(0.75)-x.quantile(0.25))))

    # Aggregation per Molecule
    per_mol = (per_assay
        .groupby(["molecule_chembl_id", "std_canonical_smiles"], as_index=False)
        .agg(pIC50=("pIC50", "median"),
             n_assays=("assay_id", "nunique"),
             n_total=("n_repl", "sum"),
             iqr_median=("pIC50", robust_mad)))

    qc["n_per_assay"] = len(per_assay)
    qc["n_per_mol"] = len(per_mol)
    return per_assay, per_mol, qc

# ==============================================================================
# 4) IO & FETCHING LOGIC
# ==============================================================================
def save_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def target_paths(target_id: str):
    base = TARGET_BASE_DIR / target_id / "chembl_data"
    return {
        "raw": base / f"{target_id}_raw_{ASSAY_TYPE_TO_PROCESS}.csv",
        "per_assay": base / f"{target_id}_pIC50_by_assay_{ASSAY_TYPE_TO_PROCESS}.csv",
        "per_mol": base / f"{target_id}_pIC50_{ASSAY_TYPE_TO_PROCESS}.csv",
        "qc": base / f"{target_id}_qc_{ASSAY_TYPE_TO_PROCESS}.json",
    }

def fetch_chembl_activities(target_id: str) -> pd.DataFrame:
    activity_client = new_client.activity
    res = activity_client.filter(target_chembl_id=target_id).filter(standard_type=ASSAY_TYPE_TO_PROCESS)
    return pd.DataFrame.from_dict(res)

def retrive_clean_and_cache(target_id: str):
    paths = target_paths(target_id)

    # If processed file exists, return it (to avoid re-downloading/re-processing)
    if paths["per_mol"].is_file():
        print(f"[{target_id}] Processed file found. Loading from cache.")
        per_mol = pd.read_csv(paths["per_mol"])
        try:
            per_assay = pd.read_csv(paths["per_assay"])
        except Exception:
            per_assay = pd.DataFrame()
        try:
            with open(paths["qc"], "r") as f:
                qc = json.load(f)
        except Exception:
            qc = {"note": "qc missing"}
        return per_assay, per_mol, qc

    # Check for raw file
    if not paths["raw"].is_file():
        if USE_CACHE_ONLY:
            return pd.DataFrame(), pd.DataFrame(), {"error": "cache_only and no raw"}
        print(f"[{target_id}] Downloading ChEMBL activities...")
        try:
            reports = fetch_chembl_activities(target_id)
            save_csv(reports, paths["raw"])
        except Exception as e:
            print(f"[{target_id}] Error downloading: {e}")
            return pd.DataFrame(), pd.DataFrame(), {"error": str(e)}
    else:
        print(f"[{target_id}] Raw file found. Loading...")
        reports = pd.read_csv(paths["raw"], low_memory=False)

    print(f"[{target_id}] Processing and Standardizing...")
    per_assay, per_mol, qc = preprocess_to_pIC50_tables(reports)
    
    # Save processed files
    save_csv(per_assay, paths["per_assay"])
    save_csv(per_mol, paths["per_mol"])
    with open(paths["qc"], "w") as f:
        json.dump(qc, f, indent=2)
        
    print(f"[{target_id}] Done. Molecules: {qc.get('n_per_mol', 0)}")
    return per_assay, per_mol, qc

# ==============================================================================
# 5) MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    print(f"Starting Download/Processing for {len(CHEMBL_TARGET_IDS)} targets...")
    
    # Sequential execution (simplest for data download part)
    # If parallel is needed, use ProcessPoolExecutor similar to original script
    results = []
    for tid in tqdm(CHEMBL_TARGET_IDS):
        try:
            _, df_mol, qc_data = retrive_clean_and_cache(tid)
            results.append({
                "Target": tid, 
                "Status": "OK" if not df_mol.empty else "Empty/Fail",
                "Count": len(df_mol) if not df_mol.empty else 0
            })
        except Exception as e:
            results.append({"Target": tid, "Status": f"Error: {e}", "Count": 0})
            
    print("\nSummary:")
    print(pd.DataFrame(results))

## Molecular fingerprints

In [ ]:
import pandas as pd
import numpy as np
import importlib
import urllib.request
from pathlib import Path
from tqdm.auto import tqdm

# ── RDKit & Chemoinformatics ──────────────────────────────────────────────────
from rdkit import Chem, Descriptors
from rdkit.Chem import AllChem, EState
from rdkit.Chem.Crippen import MolLogP, MolMR
from rdkit.Chem.rdMolDescriptors import GetUSRCAT

# ── External Libraries ────────────────────────────────────────────────────────
# Ensure you have installed: pip install mordred mol2vec gensim
from mordred import Calculator, descriptors
from mol2vec.features import MolSentence, mol2alt_sentence
from gensim.models import word2vec

# ==============================================================================
# 1) CONFIGURATION
# ==============================================================================
BASE_DIR = Path(__file__).resolve().parent
TARGET_BASE_DIR = BASE_DIR
MAX_ATOMS_FOR_MORDRED = 120  # Optimization: Skip massive molecules for 3D/Complex desc

# Full list of targets from your original Main.py
CHEMBL_TARGET_IDS = [
    "CHEMBL3105", "CHEMBL1824", "CHEMBL4005", "CHEMBL3130", "CHEMBL3267",
    "CHEMBL3145", "CHEMBL4282", "CHEMBL2842", "CHEMBL3650", "CHEMBL2742",
    "CHEMBL1871", "CHEMBL203", "CHEMBL1957", "CHEMBL4630", "CHEMBL279",
    "CHEMBL267", "CHEMBL4722", "CHEMBL2185", "CHEMBL325", "CHEMBL1865",
]

# ── Setup Mol2Vec ─────────────────────────────────────────────────────────────
MOL2VEC_MODEL_PATH = BASE_DIR / "model_300dim.pkl"
if not MOL2VEC_MODEL_PATH.exists():
    print("Downloading Mol2Vec pre-trained model...")
    url = "https://github.com/samoturk/Mol2Vec/blob/master/examples/models/model_300dim.pkl?raw=true"
    try:
        urllib.request.urlretrieve(url, str(MOL2VEC_MODEL_PATH))
        print("Download complete.")
    except Exception as e:
        print(f"Error downloading Mol2Vec model: {e}")

try:
    mol2vec_model = word2vec.Word2Vec.load(str(MOL2VEC_MODEL_PATH))
except Exception:
    mol2vec_model = None
    print("WARNING: Mol2Vec model could not be loaded. Mol2Vec descriptors will be skipped.")

# ── Setup Mordred ─────────────────────────────────────────────────────────────
# We import specific submodules to allow dynamic 2D vs 3D calculation
_desc_Chi    = importlib.import_module("mordred.chi")
_desc_Kappa  = importlib.import_module("mordred.kappa")
_desc_EState = importlib.import_module("mordred.estate")
_desc_WHIM   = importlib.import_module("mordred.whim")
_desc_RDF    = importlib.import_module("mordred.rdf")
_desc_RMSD   = importlib.import_module("mordred.rmsd")

# 2D Calculator (Fast)
mordred_calc_2d = Calculator([_desc_Chi, _desc_Kappa, _desc_EState], ignore_3D=True)
# 3D Calculator (Slower, requires embedding)
mordred_calc_3d = Calculator([_desc_WHIM, _desc_RDF, _desc_RMSD], ignore_3D=False)


# ==============================================================================
# 2) HELPER FUNCTIONS
# ==============================================================================
def load_target_data(target_id):
    """Loads the cleaned pIC50 CSV generated in step 1."""
    path = TARGET_BASE_DIR / target_id / "chembl_data" / f"{target_id}_pIC50_IC50.csv"
    if not path.exists():
        print(f"  [!] Skipped {target_id}: File not found ({path})")
        return None
    
    df = pd.read_csv(path)
    # Reconstruct RDKit Mol objects
    smi_col = "std_canonical_smiles" if "std_canonical_smiles" in df.columns else "canonical_smiles"
    df["Mol"] = df[smi_col].apply(Chem.MolFromSmiles)
    
    # Drop rows where molecule creation failed
    df = df.dropna(subset=["Mol"]).reset_index(drop=True)
    df["num_atoms"] = df["Mol"].apply(lambda m: m.GetNumAtoms())
    return df

def save_descriptor_csv(df_source, df_desc, target_id, name):
    """Saves the calculated descriptors side-by-side with ID and pIC50."""
    save_dir = TARGET_BASE_DIR / target_id / "chembl_data"
    save_dir.mkdir(parents=True, exist_ok=True)
    path = save_dir / f"{target_id}_pIC50_{name}.csv"
    
    # Metadata columns to keep
    meta = df_source[["molecule_chembl_id", "std_canonical_smiles", "pIC50"]].reset_index(drop=True)
    out = pd.concat([meta, df_desc.reset_index(drop=True)], axis=1)
    
    out.to_csv(path, index=False)
    # print(f"    -> Saved {name}")

def _embed_3d(m):
    """Generates a 3D conformer with a fixed seed."""
    m3 = Chem.AddHs(m)
    params = AllChem.ETKDG()
    params.randomSeed = 0xF00D
    if AllChem.EmbedMolecule(m3, params) == 0:
        return m3
    return None

def check_exists(target_id, name):
    save_dir = TARGET_BASE_DIR / target_id / "chembl_data"
    return (save_dir / f"{target_id}_pIC50_{name}.csv").exists()

# ==============================================================================
# 3) DESCRIPTOR GENERATION LOGIC
# ==============================================================================

def run_rdkit_desc(df, target_id):
    # 1. PhysChem
    if not check_exists(target_id, "RDKit_PhysChem"):
        try:
            names = [n for n, _ in Descriptors._descList]
            # Calculate all available RDKit descriptors
            vals = [[f(m) for _, f in Descriptors._descList] for m in df["Mol"]]
            save_descriptor_csv(df, pd.DataFrame(vals, columns=names), target_id, "RDKit_PhysChem")
        except Exception as e: print(f"    [!] PhysChem Error: {e}")

    # 2. Extended
    if not check_exists(target_id, "RDKit_Extended_Desc"):
        try:
            logp = [MolLogP(m) for m in df["Mol"]]
            mr = [MolMR(m) for m in df["Mol"]]
            es = [float(sum(EState.EStateIndices(m))) for m in df["Mol"]]
            ext = pd.DataFrame({"RDKit_LogP": logp, "RDKit_MR": mr, "RDKit_EState_Sum": es})
            save_descriptor_csv(df, ext, target_id, "RDKit_Extended_Desc")
        except Exception as e: print(f"    [!] Extended Error: {e}")

def run_lingo(df, target_id):
    if check_exists(target_id, "LINGO_Kmers"): return
    try:
        k_lens = [3, 4, 5]
        out = pd.DataFrame(index=df.index)
        smiles = df["std_canonical_smiles"].tolist()
        
        for k in k_lens:
            bags = []
            for s in smiles:
                bag = {}
                for i in range(len(s)-k+1):
                    sub = s[i:i+k]
                    bag[sub] = bag.get(sub, 0) + 1
                bags.append(bag)
            tmp = pd.DataFrame(bags).fillna(0)
            tmp.columns = [f"LINGO_K{k}_{c}" for c in tmp.columns]
            # Keep top 100 features per K to save space
            if not tmp.empty:
                keep = tmp.sum().nlargest(100).index
                out = pd.concat([out, tmp[keep]], axis=1)
        
        save_descriptor_csv(df, out, target_id, "LINGO_Kmers")
    except Exception as e: print(f"    [!] LINGO Error: {e}")

def run_mol2vec(df, target_id):
    if not mol2vec_model or check_exists(target_id, "Mol2Vec_Embeddings"): return
    try:
        sentences = [MolSentence(mol2alt_sentence(m, 1)) for m in df["Mol"]]
        vecs = []
        for s in sentences:
            wv = [mol2vec_model.wv[w] for w in s.sentence if w in mol2vec_model.wv.key_to_index]
            vecs.append(np.mean(wv, axis=0) if wv else np.zeros(300))
        
        df_m2v = pd.DataFrame(np.vstack(vecs), columns=[f"Mol2Vec_{i}" for i in range(300)])
        save_descriptor_csv(df, df_m2v, target_id, "Mol2Vec_Embeddings")
    except Exception as e: print(f"    [!] Mol2Vec Error: {e}")

def run_mordred(df, target_id):
    # Filter: Mordred can hang on very large molecules
    sub = df[df["num_atoms"] <= MAX_ATOMS_FOR_MORDRED].copy()
    if sub.empty: return

    # 1. Mordred 2D
    if not check_exists(target_id, "Mordred_2D_Chi_Kappa_EState"):
        try:
            mols = sub["Mol"].tolist()
            d2 = mordred_calc_2d.pandas(mols, nproc=1).apply(pd.to_numeric, errors='coerce')
            save_descriptor_csv(sub, d2, target_id, "Mordred_2D_Chi_Kappa_EState")
        except Exception as e: print(f"    [!] Mordred 2D Error: {e}")

    # 2. Mordred 3D (Requires Embedding)
    if not check_exists(target_id, "Mordred_3D_4D_WHIM_RDF_RMSD"):
        try:
            mols_3d = []
            valid_idxs = []
            for i, m in enumerate(sub["Mol"]):
                m3 = _embed_3d(m)
                if m3:
                    mols_3d.append(m3)
                    valid_idxs.append(sub.index[i])
            
            if mols_3d:
                d3 = mordred_calc_3d.pandas(mols_3d, nproc=1).apply(pd.to_numeric, errors='coerce')
                save_descriptor_csv(sub.loc[valid_idxs], d3, target_id, "Mordred_3D_4D_WHIM_RDF_RMSD")
        except Exception as e: print(f"    [!] Mordred 3D Error: {e}")

def run_usrcat(df, target_id):
    if check_exists(target_id, "Spectrophore_USRCAT"): return
    try:
        mols_3d = []
        valid_idxs = []
        vals = []
        
        # USRCAT requires 3D coordinates
        for i, m in enumerate(df["Mol"]):
            m3 = _embed_3d(m)
            if m3:
                vals.append(np.array(GetUSRCAT(m3)).flatten())
                valid_idxs.append(df.index[i])
                
        if vals:
            cols = [f"USRCAT_{i}" for i in range(60)]
            out = pd.DataFrame(np.vstack(vals), columns=cols)
            save_descriptor_csv(df.loc[valid_idxs], out, target_id, "Spectrophore_USRCAT")
    except Exception as e: print(f"    [!] USRCAT Error: {e}")

# ==============================================================================
# 4) MAIN EXECUTION LOOP
# ==============================================================================
if __name__ == "__main__":
    print(f"Starting Advanced Descriptor Generation for {len(CHEMBL_TARGET_IDS)} targets...")
    
    for tid in tqdm(CHEMBL_TARGET_IDS, desc="Targets"):
        # 1. Load Data
        df = load_target_data(tid)
        if df is None: 
            continue
            
        print(f"Processing {tid} ({len(df)} mols)...")
        
        # 2. Run Generators
        # Comment out lines if you don't need specific descriptors
        run_rdkit_desc(df, tid)
        run_lingo(df, tid)
        run_mol2vec(df, tid)
        run_mordred(df, tid)
        run_usrcat(df, tid)

    print("\n--- All targets processed ---")

## Feature Selection

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

# ==============================================================================
# 1. Custom Correlation Filter
#    "High correlated features... Pearson correlation coefficient > 0.95 
#     were identified and removed"
# ==============================================================================

class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Filter features that are highly correlated with each other.
    Keeps the first feature encountered and drops subsequent highly correlated ones.
    """
    def __init__(self, threshold: float = 0.95, random_state: int = None):
        self.threshold = threshold
        self.random_state = random_state
        self.to_drop_ = None

    def fit(self, X, y=None):
        # Conversion to DataFrame handles column tracking better, 
        # but assumes X is numpy array in pipeline.
        if isinstance(X, np.ndarray):
            df = pd.DataFrame(X)
        else:
            df = pd.DataFrame(X).copy()
            
        # Calculate Pearson correlation matrix
        corr_matrix = df.corr().abs()
        
        # Select upper triangle of correlation matrix
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
        # Identify columns to drop (corr > threshold)
        self.to_drop_ = [column for column in upper.columns if any(upper[column] > self.threshold)]
        return self

    def transform(self, X):
        if self.to_drop_ is None:
            return X
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.to_drop_)
        # If numpy, we need indices (mapped from fit step)
        return np.delete(X, self.to_drop_, axis=1)

# ==============================================================================
# 2. Pipeline Construction
#    "Zero variance filter... custom correlation filter... 
#     Standardized using Z-score normalization (except for tree-based models)"
# ==============================================================================

def get_preprocessing_pipelines(random_seed=42):
    
    # Common filtering pipeline:
    # 1. VarianceThreshold(0.0): Removes zero variance features.
    # 2. CorrelationFilter: Removes features with corr > 0.95.
    feature_selection_pipe = Pipeline([
        ('variance', VarianceThreshold(0.0)), 
        ('corr', CorrelationFilter(threshold=0.95, random_state=random_seed))
    ])

    scaler = StandardScaler()

    # --- Scenario A: Models Sensitive to Scaling (e.g., Linear, SVM, KNN) ---
    # Includes StandardScaler
    linear_pipeline = Pipeline([
        ('scaler', scaler),             # Z-score Normalization
        ('fs', feature_selection_pipe), # Feature Selection
        ('est', LinearRegression())     # Estimator
    ])

    # --- Scenario B: Tree-Based Models (Invariant to Scaling) ---
    # SKIPS StandardScaler
    tree_pipeline = Pipeline([
        ('fs', feature_selection_pipe),             # Feature Selection Only
        ('est', RandomForestRegressor(n_jobs=1))    # Estimator
    ])
    
    return linear_pipeline, tree_pipeline

## Training and Testing

In [ ]:
import os
import sys
import time
import json
import warnings
import traceback
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, Tuple, List, Any

# ─── Parallel Processing & Serialization ──────────────────────────────────────
from joblib import Parallel, delayed, dump, load

# ─── Chemoinformatics (RDKit) ─────────────────────────────────────────────────
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

# ─── Scikit-Learn Ecosystem ───────────────────────────────────────────────────
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectPercentile, SelectFromModel, f_regression
from sklearn.model_selection import GroupKFold, HalvingRandomSearchCV, RandomizedSearchCV
from sklearn.metrics import r2_score
from sklearn.compose import TransformedTargetRegressor

# ─── Models ───────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor

# Try importing XGBoost (External dependency)
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# Suppress warnings for cleaner logs
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ==============================================================================
# 1. PREPROCESSING HELPERS
# ==============================================================================

class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Custom Transformer: Drops features with Pearson correlation > threshold.
    """
    def __init__(self, threshold: float = 0.95):
        self.threshold = threshold
        self.to_drop_ = None

    def fit(self, X, y=None):
        if isinstance(X, np.ndarray):
            df = pd.DataFrame(X)
        else:
            df = pd.DataFrame(X).copy()
            
        corr_matrix = df.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.to_drop_ = [column for column in upper.columns if any(upper[column] > self.threshold)]
        return self

    def transform(self, X):
        if self.to_drop_ is None:
            return X
        if isinstance(X, pd.DataFrame):
            return X.drop(columns=self.to_drop_)
        return np.delete(X, self.to_drop_, axis=1)

def make_scaffold_groups(smiles_list: List[str]) -> List[str]:
    """Generates Murcko Scaffold strings for splitting."""
    scaffolds = []
    for s in smiles_list:
        try:
            m = Chem.MolFromSmiles(s or '')
            if m:
                scaf = MurckoScaffold.GetScaffoldForMol(m)
                scaffolds.append(Chem.MolToSmiles(scaf, canonical=True))
            else:
                scaffolds.append('NA_SCAFFOLD')
        except:
            scaffolds.append('INVALID_SMILES')
    return scaffolds

def get_holdout_mask(df, holdout_frac=0.15, seed=42):
    """Creates a True/False mask for the External Test Set based on Scaffolds."""
    groups = df['scaffold_group'].values
    unique_groups = np.unique(groups)
    rng = np.random.RandomState(seed)
    rng.shuffle(unique_groups)
    
    mask = np.zeros(len(df), dtype=bool)
    total = len(df)
    count = 0
    min_hold = max(1, int(holdout_frac * total * 0.5))
    
    for g in unique_groups:
        if g in ['NA_SCAFFOLD', 'INVALID_SMILES']: continue
        idx = np.where(groups == g)[0]
        mask[idx] = True
        count += len(idx)
        if count / total >= holdout_frac and count >= min_hold:
            break
            
    # Fallback to random if scaffolds fail
    if mask.sum() < min_hold:
        choice = np.random.RandomState(seed + 999).choice(
            df.index, size=max(min_hold, int(holdout_frac*total)), replace=False
        )
        mask = df.index.isin(choice)
    return mask

# ==============================================================================
# 2. WORKER FUNCTION (Executes 1 Task)
# ==============================================================================

def train_and_evaluate_task(
    config: Dict[str, Any],
    dataset_name: str,
    representation: str,
    model_name: str,
    pipeline: Pipeline,
    param_grid: List[Dict]
) -> Dict[str, Any]:
    
    result_payload = {
        'status': 'fail',
        'dataset': dataset_name,
        'representation': representation,
        'model': model_name,
        'ext_result': None,
        'error_info': None
    }

    try:
        base_dir = Path(config['BASE_DIR'])
        out_dir = Path(config['OUT_DIR'])
        
        # --- 1. Load Data ---
        data_dir = base_dir / dataset_name / 'chembl_data'
        base_csv = data_dir / f"{dataset_name}_pIC50_{config['ASSAY']}.csv"
        rep_csv = data_dir / f"{dataset_name}_pIC50_{representation}.csv"
        
        # Merge ID/Activity with Descriptors
        base_df = pd.read_csv(base_csv, usecols=["molecule_chembl_id", "std_canonical_smiles", "pIC50"]).dropna(subset=['pIC50'])
        rep_df = pd.read_csv(rep_csv)
        df = pd.merge(base_df, rep_df, on=["molecule_chembl_id", "std_canonical_smiles"], how='inner')
        
        if 'pIC50_x' in df.columns: # Handle duplicate columns
            df['pIC50'] = df['pIC50_x']
            df = df.drop(columns=['pIC50_x', 'pIC50_y'], errors='ignore')

        # Clean Features
        exclude_cols = {"molecule_chembl_id", "std_canonical_smiles", "pIC50", 'scaffold_group'}
        feature_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
        if len(feature_cols) < 2: raise ValueError("Not enough feature columns found.")
        
        # Simple Imputation (Inf/NaN -> 0)
        df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        
        # --- 2. Scaffold Split ---
        df['scaffold_group'] = make_scaffold_groups(df['std_canonical_smiles'].tolist())
        holdout_mask = get_holdout_mask(df, config['HOLDOUT_FRAC'], config['RNG_SEED'])
        
        train_df = df.loc[~holdout_mask].reset_index(drop=True)
        test_df  = df.loc[holdout_mask].reset_index(drop=True)
        
        X_train = train_df[feature_cols].values
        y_train = train_df['pIC50'].values
        g_train = train_df['scaffold_group'].values
        
        X_test  = test_df[feature_cols].values
        y_test  = test_df['pIC50'].values
        
        # --- 3. Hyperparameter Tuning & Training ---
        t0 = time.time()
        
        # Setup CV
        cv = GroupKFold(n_splits=config['INNER_FOLDS'])
        splits = list(cv.split(X_train, y_train, g_train))
        
        est = clone(pipeline)
        
        if any(param_grid):
            # Use HalvingRandomSearchCV for speed, or fallback to RandomizedSearchCV
            if config['USE_HALVING_SEARCH']:
                search = HalvingRandomSearchCV(
                    est, param_grid, n_candidates=config['N_ITER_SEARCH'],
                    cv=splits, factor=3, min_resources='exhaust',
                    random_state=config['RNG_SEED'], n_jobs=1, verbose=0, error_score='raise'
                )
            else:
                search = RandomizedSearchCV(
                    est, param_grid, n_iter=config['N_ITER_SEARCH'],
                    cv=splits, random_state=config['RNG_SEED'], n_jobs=1, verbose=0
                )
            search.fit(X_train, y_train)
            best_est = search.best_estimator_
        else:
            # No params to tune (e.g., standard LinearRegression)
            best_est = est.fit(X_train, y_train)
            
        train_time = time.time() - t0
        
        # --- 4. External Evaluation ---
        y_pred = best_est.predict(X_test)
        r2_ext = r2_score(y_test, y_pred)
        
        # --- 5. Save Artifacts ---
        # A. Predictions
        pred_df = test_df[['molecule_chembl_id', 'std_canonical_smiles', 'pIC50']].copy()
        pred_df['predicted_pIC50'] = y_pred
        pred_path = out_dir / f"pred_{dataset_name}_{representation}_{model_name}.csv"
        pred_df.to_csv(pred_path, index=False)
        
        # B. Model
        model_dir = out_dir / 'models'
        model_dir.mkdir(exist_ok=True)
        dump(best_est, model_dir / f"{dataset_name}_{representation}_{model_name}.joblib", compress=3)
        
        result_payload['status'] = 'success'
        result_payload['ext_result'] = {
            'dataset': dataset_name,
            'representation': representation,
            'model': model_name,
            'R2': r2_ext,
            'train_time': train_time
        }
        
    except Exception as e:
        result_payload['error_info'] = {'msg': str(e), 'trace': traceback.format_exc()}
        
    return result_payload

# ==============================================================================
# 3. MAIN RUNNER (Configuration & Model Zoo)
# ==============================================================================

class QSARRunner:
    # --- Configuration ---
    BASE_DIR = Path('../Documents/ChEMBL_data')  # Change to your path
    OUT_DIR = Path('./QSAR_Results')
    ASSAY = 'IC50'
    
    USE_GPU = False
    RNG_SEED = 42
    HOLDOUT_FRAC = 0.15
    OUTER_FOLDS = 5 # (Not used in this simpler Holdout script, but good for CV)
    INNER_FOLDS = 3 # For Hyperparam tuning
    N_ITER_SEARCH = 20 # Number of hyperparam combinations to try
    USE_HALVING_SEARCH = True
    
    # Representations to process
    REPRESENTATIONS = [
        "ECFP4", "ECFP6", "MACCS", "AtomPair", "Torsion",
        "RDKit_PhysChem", "RDKit_Extended_Desc", 
        "Mordred_2D_Chi_Kappa_EState", "Mol2Vec_Embeddings"
    ]

    def __init__(self):
        self.OUT_DIR.mkdir(parents=True, exist_ok=True)
        self.summary_path = self.OUT_DIR / "summary_results.csv"

    def _get_model_zoo(self) -> Dict[str, Tuple[Pipeline, List[Dict]]]:
        """
        DEFINES ALL MODELS AND HYPERPARAMETER GRIDS.
        """
        scaler = StandardScaler()
        # Base Filtering: Remove Zero Variance & High Correlation
        fixed_pipe = Pipeline([
            ('variance', VarianceThreshold(0.0)), 
            ('corr', CorrelationFilter(threshold=0.95))
        ])
        
        # Feature Selection Grid (Lasso vs F_Regression vs None)
        fs_pipe = Pipeline([('fixed', fixed_pipe), ('selector', 'passthrough')])
        
        fs_spaces = [
            {'selector': ['passthrough']},
            {'selector': [SelectPercentile(score_func=f_regression)], 'selector__percentile': [25, 50, 75]},
            {'selector': [SelectFromModel(Lasso(alpha=0.01, random_state=self.RNG_SEED))], 'selector__threshold': ['median', 'mean']}
        ]
        
        def combine(est_grid, fs_grids=fs_spaces):
            """Combines estimator params with feature selection params"""
            return [
                {**e, **{f'fs__{k}': v for k, v in f.items()}} 
                for e in (est_grid if isinstance(est_grid, list) else [est_grid or {}])
                for f in fs_grids
            ]

        M = {}
        
        M['Ridge'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', Ridge(random_state=self.RNG_SEED))]), 
            combine({'est__alpha': np.logspace(-4, 4, 20)})
        )
        M['ElasticNet'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', ElasticNet(random_state=self.RNG_SEED))]), 
            combine({'est__alpha': np.logspace(-4, 2, 10), 'est__l1_ratio': [0.1, 0.5, 0.9]})
        )
        M['BayesianRidge'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', BayesianRidge())]), 
            combine({})
        )
        
        # --- 2. Support Vector Machines (Require Scaling) ---
        M['SVR_rbf'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', SVR(kernel='rbf'))]), 
            combine({'est__C': np.logspace(-2, 3, 5), 'est__gamma': np.logspace(-4, 0, 5)})
        ) 
        # --- 3. Neighbors (Require Scaling) ---
        M['KNN'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', KNeighborsRegressor())]), 
            combine({'est__n_neighbors': [3, 5, 7, 11, 15]})
        )
        
        # --- 4. Tree Ensembles (No Scaling Needed) ---
        M['RandomForest'] = (
            Pipeline([('fs', fs_pipe), ('est', RandomForestRegressor(n_jobs=1, random_state=self.RNG_SEED))]), 
            combine({'est__n_estimators': [100, 300], 'est__max_features': ['sqrt', 'log2']})
        )
        M['ExtraTrees'] = (
            Pipeline([('fs', fs_pipe), ('est', ExtraTreesRegressor(n_jobs=1, random_state=self.RNG_SEED))]), 
            combine({'est__n_estimators': [100, 300], 'est__max_features': ['sqrt', 'log2']})
        )
        M['HistGBDT'] = (
            Pipeline([('fs', fs_pipe), ('est', HistGradientBoostingRegressor(random_state=self.RNG_SEED))]), 
            combine({'est__max_iter': [200, 500], 'est__learning_rate': [0.01, 0.05, 0.1, 0.2]})
        )
        
        if HAS_XGB:
            M['XGBoost'] = (
                Pipeline([('fs', fs_pipe), ('est', XGBRegressor(n_jobs=1, random_state=self.RNG_SEED, verbosity=0))]), 
                combine({'est__n_estimators': [200, 500], 'est__learning_rate': [0.01, 0.05, 0.1], 'est__max_depth': [3, 6, 9]})
            )

        # --- 5. Others ---
        M['PLSRegression'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', PLSRegression())]), 
            combine({'est__n_components': [2, 5, 10]})
        )
        M['MLP'] = (
            Pipeline([('scaler', scaler), ('fs', fs_pipe), ('est', MLPRegressor(random_state=self.RNG_SEED, max_iter=500))]), 
            combine({'est__hidden_layer_sizes': [(50,), (100,), (100, 50)], 'est__alpha': [1e-4, 1e-2]})
        )

        return M

    def run(self):
        print(f"--- Starting QSAR Pipeline [Output: {self.OUT_DIR}] ---")
        
        # Discover Datasets
        targets = []
        for p in self.BASE_DIR.glob('CHEMBL*'):
            if (p / 'chembl_data' / f"{p.name}_pIC50_{self.ASSAY}.csv").exists():
                targets.append(p.name)
        
        if not targets:
            print("No datasets found!")
            return

        model_zoo = self._get_model_zoo()
        print(f"Found {len(targets)} targets. Models: {list(model_zoo.keys())}")
        
        # Create Task List
        tasks = []
        for target in targets:
            # Check available descriptors for this target
            avail_reps = []
            for r in self.REPRESENTATIONS:
                if (self.BASE_DIR / target / 'chembl_data' / f"{target}_pIC50_{r}.csv").exists():
                    avail_reps.append(r)
            
            for rep in avail_reps:
                for model_name, (pipe, params) in model_zoo.items():
                    # Check if already done
                    if (self.OUT_DIR / f"pred_{target}_{rep}_{model_name}.csv").exists():
                        continue
                        
                    tasks.append((target, rep, model_name, pipe, params))

        print(f"Total tasks to run: {len(tasks)}")
        if not tasks:
            print("All tasks completed.")
            return

        # Prepare Config Dictionary for Workers
        config = {
            'BASE_DIR': str(self.BASE_DIR),
            'OUT_DIR': str(self.OUT_DIR),
            'ASSAY': self.ASSAY,
            'RNG_SEED': self.RNG_SEED,
            'HOLDOUT_FRAC': self.HOLDOUT_FRAC,
            'INNER_FOLDS': self.INNER_FOLDS,
            'N_ITER_SEARCH': self.N_ITER_SEARCH,
            'USE_HALVING_SEARCH': self.USE_HALVING_SEARCH
        }

        # Run Parallel
        n_cores = max(1, os.cpu_count() - 2)
        print(f"Running on {n_cores} cores...")
        
        results = Parallel(n_jobs=n_cores, verbose=5)(
            delayed(train_and_evaluate_task)(config, *t) for t in tasks
        )
        
        # Process Results
        summary_list = []
        for res in results:
            if res['status'] == 'success':
                summary_list.append(res['ext_result'])
            else:
                print(f"FAIL: {res['dataset']}-{res['model']} -> {res['error_info']['msg']}")
        
        if summary_list:
            df = pd.DataFrame(summary_list)
            header = not self.summary_path.exists()
            df.to_csv(self.summary_path, mode='a', header=header, index=False)
            print(f"Saved summary to {self.summary_path}")

# ==============================================================================
# 4. EXECUTION
# ==============================================================================
if __name__ == '__main__':
    runner = QSARRunner()
    runner.run()

## Statistical Analysis